[&#8592; Previous: Supervised Learning](supervised_learning.ipynb)

# **Supervised Learning Extensions**

This notebook **extends `supervised_learning.ipynb`**. The main notebook answers the core
question (does county MIPS performance add predictive value for county Medicare outcomes beyond
demographics) and its findings stand on their own. The two extensions here probe how the answer
changes under two deliberate refinements:

- **Extension A, population weighting.** The main analysis treats every county equally, so a
  1,000-beneficiary county counts as much as a 1,000,000-beneficiary one. Weighting by beneficiary
  count is standard in county-level health services work: it reduces small-county noise and answers
  the question "does the relationship hold where most beneficiaries actually live."
- **Extension B, PCA features.** The 49 MIPS measures are correlated. Replacing them with
  principal components (fit inside each training fold) removes multicollinearity by construction
  and ties this section directly to the unsupervised notebook, which found the same latent
  performance dimensions.

> **Note:** run `supervised_learning.ipynb` first. It exports the modeling frame and feature
> metadata this notebook loads, so both notebooks model the exact same panel.

**Setup:** Pinned dependencies: `python -m pip install -r ../requirements.txt`

## Table of Contents

1. [Setup and load the modeling frame](#se1)
2. [Extension A: population-weighted models](#se2)
3. [Extension B: PCA feature variant](#se3)
4. [Summary](#se4)

In [1]:
# Install dependencies (remove -q to see output)
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


---
<a id="se1"></a>

## **1. Setup and load the modeling frame**

Everything comes from the main notebook's export: same counties, same 49 screened measures, same
controls, same fold construction (the frame is sorted by county FIPS, so GroupKFold reproduces the
identical splits).

In [2]:
# Standard Library
import json
from pathlib import Path

# Data Processing
import numpy as np
import polars as pl

# Machine Learning
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score

DATA = Path("../data")
RANDOM_STATE = 42
MODEL_YEAR = "2023"
N_SPLITS = 5
PRIMARY_TARGET = "ACUTE_HOSP_READMSN_PCT"

# Exported by supervised_learning.ipynb (run that first)
frame = pl.read_parquet(DATA / "interim" / f"modeling_frame_{MODEL_YEAR}.parquet")
meta = json.loads((DATA / "interim" / f"modeling_metadata_{MODEL_YEAR}.json").read_text())
CONTROLS = meta["controls"]
final_measures = meta["final_measures"]
FEATURE_SETS = meta["feature_sets"]

assert frame.height == 2507, f"unexpected frame size: {frame.height}"
assert len(final_measures) == 49

sub = frame.filter(pl.col(PRIMARY_TARGET).is_not_null()).sort("county_fips")
X_all = sub.select(FEATURE_SETS["controls_plus_mips"]).to_pandas()
y = sub[PRIMARY_TARGET].to_numpy()
groups = sub["state"].to_numpy()
weights = np.exp(sub["log_benes"].to_numpy())
folds = list(GroupKFold(n_splits=N_SPLITS).split(X_all, y, groups))

MODELS = {
    "linear": lambda: LinearRegression(),
    "elastic_net": lambda: ElasticNetCV(l1_ratio=[0.1, 0.5, 0.9, 1.0], alphas=50,
                                        max_iter=5000, random_state=RANDOM_STATE),
    "random_forest": lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                                   n_jobs=-1, random_state=RANDOM_STATE),
    "grad_boost": lambda: HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

def make_pipe(model):
    """Wrap an estimator with the shared preprocessing steps.

    Args:
        model: An unfitted sklearn estimator.

    Returns:
        Pipeline of median imputation, standardization, then the model, so
        preprocessing is fitted inside each training fold only.
    """
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("model", model),
    ])

print(f"panel: {len(y):,} counties, {len(final_measures)} measures, {sub['state'].n_unique()} states")
print(f"beneficiary counts: min={weights.min():,.0f}  median={np.median(weights):,.0f}  max={weights.max():,.0f}")

panel: 2,458 counties, 49 measures, 51 states
beneficiary counts: min=211  median=9,175  max=1,678,944


---
<a id="se2"></a>

## **2. Extension A: population-weighted models**

Same models, same folds, but each county's contribution to the model fit is weighted by its
beneficiary population (imputation and scaling statistics stay per-county by design, so feature
definitions match the main notebook exactly). Test-fold R2 stays unweighted so the numbers are
directly comparable. If the MIPS lift survives weighting, it is not an artifact of noisy small
counties. As in the main notebook, ElasticNetCV tunes its penalty with an inner ungrouped K-fold
on training rows only; outer scores are still computed on unseen states.

In [3]:
def weighted_cv(model_factory, X, y, w, folds):
    """Cross-validate a pipeline with per-county sample weights.

    Args:
        model_factory: Zero-argument callable returning an unfitted estimator.
        X: Feature DataFrame.
        y: Target array.
        w: Sample weights, one per row of X (beneficiary counts here).
        folds: List of (train_idx, test_idx) pairs.

    Returns:
        Array of unweighted test-fold R2 scores, one per fold, so results
        stay comparable with the unweighted runs in the main notebook.
    """
    scores = []
    for tr, te in folds:
        pipe = make_pipe(model_factory())
        pipe.fit(X.iloc[tr], y[tr], model__sample_weight=w[tr])
        scores.append(r2_score(y[te], pipe.predict(X.iloc[te])))
    return np.array(scores)

records = []
for m_name, m_factory in MODELS.items():
    r2 = {}
    for fs_name, feats in FEATURE_SETS.items():
        # Unweighted reference (mirrors the main notebook)
        cv_u = cross_validate(make_pipe(m_factory()), X_all[feats], y, cv=folds,
                              scoring={"r2": "r2"}, error_score="raise")
        # Weighted fit, unweighted scoring
        r2[fs_name] = {
            "unweighted": cv_u["test_r2"].mean(),
            "weighted": weighted_cv(m_factory, X_all[feats], y, weights, folds).mean(),
        }
    for kind in ("unweighted", "weighted"):
        records.append({
            "model": m_name, "fit": kind,
            "r2_controls": r2["controls_only"][kind],
            "r2_full": r2["controls_plus_mips"][kind],
            "delta_r2": r2["controls_plus_mips"][kind] - r2["controls_only"][kind],
        })

weighted_tbl = pl.DataFrame(records)
with pl.Config(float_precision=4, tbl_rows=20):
    print(weighted_tbl)

shape: (8, 5)
┌───────────────┬────────────┬─────────────┬─────────┬──────────┐
│ model         ┆ fit        ┆ r2_controls ┆ r2_full ┆ delta_r2 │
│ ---           ┆ ---        ┆ ---         ┆ ---     ┆ ---      │
│ str           ┆ str        ┆ f64         ┆ f64     ┆ f64      │
╞═══════════════╪════════════╪═════════════╪═════════╪══════════╡
│ linear        ┆ unweighted ┆ 0.1264      ┆ 0.1171  ┆ -0.0093  │
│ linear        ┆ weighted   ┆ 0.1307      ┆ 0.1172  ┆ -0.0134  │
│ elastic_net   ┆ unweighted ┆ 0.1295      ┆ 0.1469  ┆ 0.0174   │
│ elastic_net   ┆ weighted   ┆ 0.1339      ┆ 0.1509  ┆ 0.0170   │
│ random_forest ┆ unweighted ┆ 0.1434      ┆ 0.1565  ┆ 0.0131   │
│ random_forest ┆ weighted   ┆ 0.1707      ┆ 0.1761  ┆ 0.0054   │
│ grad_boost    ┆ unweighted ┆ 0.0823      ┆ 0.1024  ┆ 0.0202   │
│ grad_boost    ┆ weighted   ┆ 0.1180      ┆ 0.1368  ┆ 0.0188   │
└───────────────┴────────────┴─────────────┴─────────┴──────────┘


---
<a id="se3"></a>

## **3. Extension B: PCA feature variant**

The 49 measures are replaced by principal components capturing 90% of their variance. The PCA is
fitted **inside each training fold** (never on test states), controls pass through untouched, and
the same fold structure applies. This mirrors the unsupervised notebook, where the same
dimensionality reduction found preventive care as the dominant axis; component-level results
should be read against its loadings.

In [4]:
def make_pca_pipe(model, controls, measures):
    """Build a pipeline that replaces the MIPS measures with principal components.

    Args:
        model: An unfitted sklearn estimator.
        controls: Control column names, passed through imputation and scaling only.
        measures: Measure column names, reduced to components explaining 90% of
            variance, fitted on training folds only.

    Returns:
        Pipeline of a ColumnTransformer (controls scaled, measures scaled then
        PCA-reduced) followed by the model.
    """
    ctrl_block = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    measure_block = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.90, random_state=RANDOM_STATE)),
    ])
    features = ColumnTransformer([
        ("controls", ctrl_block, controls),
        ("measures", measure_block, measures),
    ])
    return Pipeline([("features", features), ("model", model)])

ctrl_feats = FEATURE_SETS["controls_only"]

records = []
for m_name, m_factory in MODELS.items():
    cv_ctrl = cross_validate(make_pipe(m_factory()), X_all[ctrl_feats], y, cv=folds,
                             scoring={"r2": "r2"}, error_score="raise")
    cv_pca = cross_validate(make_pca_pipe(m_factory(), ctrl_feats, final_measures),
                            X_all, y, cv=folds, scoring={"r2": "r2"}, error_score="raise")
    records.append({
        "model": m_name,
        "r2_controls": cv_ctrl["test_r2"].mean(),
        "r2_controls_plus_pca": cv_pca["test_r2"].mean(),
        "delta_r2_pca": cv_pca["test_r2"].mean() - cv_ctrl["test_r2"].mean(),
    })

pca_tbl = pl.DataFrame(records)
with pl.Config(float_precision=4):
    print(pca_tbl)

# Reference: how many components a full-panel fit keeps at the 90% threshold
probe = make_pca_pipe(LinearRegression(), ctrl_feats, final_measures)
probe.fit(X_all, y)
n_comp = probe.named_steps["features"].named_transformers_["measures"].named_steps["pca"].n_components_
print(f"components kept at 90% variance (full-panel reference): {n_comp} of {len(final_measures)} measures")

shape: (4, 4)
┌───────────────┬─────────────┬──────────────────────┬──────────────┐
│ model         ┆ r2_controls ┆ r2_controls_plus_pca ┆ delta_r2_pca │
│ ---           ┆ ---         ┆ ---                  ┆ ---          │
│ str           ┆ f64         ┆ f64                  ┆ f64          │
╞═══════════════╪═════════════╪══════════════════════╪══════════════╡
│ linear        ┆ 0.1264      ┆ 0.1300               ┆ 0.0037       │
│ elastic_net   ┆ 0.1295      ┆ 0.1405               ┆ 0.0110       │
│ random_forest ┆ 0.1434      ┆ 0.1353               ┆ -0.0081      │
│ grad_boost    ┆ 0.0823      ┆ 0.1006               ┆ 0.0183       │
└───────────────┴─────────────┴──────────────────────┴──────────────┘
components kept at 90% variance (full-panel reference): 39 of 49 measures


---
<a id="se4"></a>

## **4. Summary**

What the two tables say against the main notebook:

- **Weighting.** The MIPS lift survives population weighting, most cleanly for Elastic Net (+0.017
  unweighted and +0.017 weighted). Gradient boosting holds at about +0.019 weighted, while random
  forest shrinks most (+0.013 unweighted to +0.005 weighted), so the flexible-tree gain is the
  least stable across the per-county and per-beneficiary views. Elastic Net is the number the report
  should lead with.
- **PCA.** About 39 components (90% variance) carry most of the lift: Elastic Net keeps +0.011 of its
  +0.017 and gradient boosting +0.018 of +0.020, while plain linear regression flips from negative to
  positive once the measures are decorrelated. That last point is the key one: it confirms the reason
  plain linear failed was multicollinearity among the correlated measures, not nonlinearity. Random
  forest slips slightly negative on components, expected tree behavior with rotated features.

**Limitations specific to these extensions**
- Weighted fits answer a per-beneficiary question, unweighted a per-county question; both are
  legitimate and the report should name which one a number comes from.
- Components are not individually named measures; interpretation routes through the unsupervised
  notebook's loadings rather than coefficients on named measures.
- Both extensions cover the primary target (readmission rate); the cost target follows the same
  machinery if needed.

In [5]:
# artifacts for the report draft
out_w = DATA / "interim" / f"extension_weighted_{MODEL_YEAR}.csv"
out_p = DATA / "interim" / f"extension_pca_{MODEL_YEAR}.csv"
weighted_tbl.write_csv(out_w)
pca_tbl.write_csv(out_p)
print(f"wrote {out_w}")
print(f"wrote {out_p}")
print("done")

wrote ../data/interim/extension_weighted_2023.csv
wrote ../data/interim/extension_pca_2023.csv
done
